<a href="https://colab.research.google.com/github/ivang9327457-cloud/solana-onchain-analytics/blob/main/solana_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import duckdb

# Download public Solana token transactions dataset directly via URL
url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/sol.csv"
print("Fetching live Solana dataset...")

# Load directly into Pandas
df = pd.read_csv(url)

# Execute SQL query using DuckDB
query = """
SELECT
    time,
    PriceUSD,
    TxCnt,
    TxTfrValAdjUSD
FROM df
WHERE PriceUSD IS NOT NULL
ORDER BY time DESC
LIMIT 10
"""

result = duckdb.query(query).df()
print("SQL Execution Success! Here are the latest Solana metrics:")
print(result)


Fetching live Solana dataset...


BinderException: Binder Error: Referenced column "PriceUSD" not found in FROM clause!
Candidate bindings: "CapMrktEstUSD", "time"

In [4]:
import pandas as pd
import duckdb

url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/sol.csv"
print("Fetching live Solana dataset...")

df = pd.read_csv(url)

# Select all available columns dynamically
query = """
SELECT *
FROM df
ORDER BY time DESC
LIMIT 5
"""

result = duckdb.query(query).df()
print("SQL Execution Success! Loaded columns:")
print(result.columns.tolist())
print("\nData Sample:")
print(result.iloc[:, :6])  # Display first 6 available columns

Fetching live Solana dataset...
SQL Execution Success! Loaded columns:
['time', 'CapMrktEstUSD', 'ReferenceRate', 'ReferenceRateBTC', 'ReferenceRateETH', 'ReferenceRateEUR', 'ReferenceRateUSD', 'volume_reported_spot_usd_1d']

Data Sample:
         time  CapMrktEstUSD  ReferenceRate  ReferenceRateBTC  \
0  2026-05-24            NaN      85.655247          0.001118   
1  2026-05-23   4.950705e+10      84.528373          0.001119   
2  2026-05-22   4.885580e+10      87.164671          0.001123   
3  2026-05-21   5.036284e+10      85.961786          0.001111   
4  2026-05-20   4.966790e+10      84.298291          0.001098   

   ReferenceRateETH  ReferenceRateEUR  
0          0.040477         73.820688  
1          0.040899         72.887700  
2          0.040881         75.040959  
3          0.040449         73.954911  
4          0.039914         72.620114  


In [5]:
import pandas as pd
import duckdb

url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/sol.csv"
df = pd.read_csv(url)

# Run full DuckDB SQL analytics pipeline
query = """
SELECT
    time,
    ROUND(ReferenceRate, 2) AS sol_price_usd,
    ROUND(CapMrktEstUSD / 1e9, 2) AS market_cap_billions,
    ROUND(ReferenceRateBTC, 6) AS sol_btc_ratio,
    ROUND(ReferenceRateETH, 6) AS sol_eth_ratio
FROM df
WHERE ReferenceRate IS NOT NULL
ORDER BY time DESC
LIMIT 10
"""

analytics_df = duckdb.query(query).df()
print("--- SOLANA ON-CHAIN ANALYTICS DASHBOARD ---")
print(analytics_df)

--- SOLANA ON-CHAIN ANALYTICS DASHBOARD ---
         time  sol_price_usd  market_cap_billions  sol_btc_ratio  \
0  2026-05-24          85.66                  NaN       0.001118   
1  2026-05-23          84.53                49.51       0.001119   
2  2026-05-22          87.16                48.86       0.001123   
3  2026-05-21          85.96                50.36       0.001111   
4  2026-05-20          84.30                49.67       0.001098   
5  2026-05-19          85.36                48.75       0.001109   
6  2026-05-18          85.26                49.36       0.001100   

   sol_eth_ratio  
0       0.040477  
1       0.040899  
2       0.040881  
3       0.040449  
4       0.039914  
5       0.040074  
6       0.039782  


In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=analytics_df['time'], y=analytics_df['sol_price_usd'], name="SOL Price (USD)", line=dict(color='#14F195', width=3)),
    secondary_y=False,
)

fig.add_trace(
    go.Bar(x=analytics_df['time'], y=analytics_df['market_cap_billions'], name="Market Cap ($B)", opacity=0.3, marker_color='#9945FF'),
    secondary_y=True,
)

fig.update_layout(title_text="Solana Network Valuation & Price Dashboard", template="plotly_dark")
fig.show()